# Treino

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import os
from collections import defaultdict
import math

# Importar módulos atualizados (Assumindo que mm_dataset_mealey.py foi atualizado)
from mm_config_mealey import CONFIG, generate_splits, DATA_ROOT, SPLIT_DIR
from mm_dataset_mealey import MMDataset 
# Assumindo que o arquivo mm_transformer_mealey.py está no mesmo diretório
from mm_transformer_mealey import MultiModalTransformer

# --- CONFIGURAÇÃO DE ESTABILIDADE (AJUSTE NOVO) ---
# Limite máximo da norma do gradiente para evitar gradientes explosivos (NAN) na GRU
MAX_GRADIENT_NORM = 1.0 # Valor típico para modelos recorrentes

# --- 0. PREPARAÇÃO INICIAL ---

# Configura o device (GPU ou CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

print("Verificando e gerando arquivos de split...")
# Garantimos que os splits existam antes de carregar
if not os.path.exists(CONFIG["train_split_path"]):
    generate_splits(DATA_ROOT, SPLIT_DIR)


# --- 1. FUNÇÕES DE TREINAMENTO E VALIDAÇÃO ---

def train_epoch(model, dataloader, criterion_pos, criterion_cls, criterion_state, optimizer):
    """
    Executa uma época de treinamento com Backpropagation Through Time (BPTT).
    Assumimos que o batch tem as chaves 'gt_state' e 'gt_drone_class' (ver seção 2 da análise).
    """
    model.train()
    running_loss = 0.0
    num_batches = len(dataloader)
    
    for batch in tqdm(dataloader, desc="Treinamento"):
        
        # O lote agora tem o formato [B, T, ...]
        B, T = batch['image'].shape[:2] 
        
        # Inicializa o estado GRU (Mealy) para o início de CADA sequência no batch (t=0)
        prev_mealy_state = None 
        
        # Listas para coletar as predições de todos os T passos
        all_pred_pos = []
        all_pred_class = []
        all_pred_state_logits = []

        optimizer.zero_grad()
        
        # LOOP PRINCIPAL DO BPTT (Iteração sobre o tempo T)
        for t in range(T):
            
            # 1. Extrai o frame t de todas as modalidades e move para o DEVICE
            image_t = batch['image'][:, t, ...].to(DEVICE)
            lidar_t = batch['lidar'][:, t, ...].to(DEVICE)
            radar_t = batch['radar'][:, t, ...].to(DEVICE)
            audio_t = batch['audio'][:, t, ...].to(DEVICE)
            
            # 2. Forward pass para o passo t (passando o estado anterior)
            output = model(image_t, lidar_t, radar_t, audio_t, prev_mealy_state)
            
            # 3. Coleta predições e estado
            all_pred_pos.append(output['pred_pos'])
            all_pred_class.append(output['pred_class'])
            all_pred_state_logits.append(output['pred_state_logits'])
            
            # 4. Propaga o estado: O estado atual se torna o estado anterior para t+1
            # Importante para BPTT
            prev_mealy_state = output['next_mealy_state'] 
            
        # --- Cálculo da Loss (Empilhando as predições de T) ---
        
        # Predições: [T, B, dim] -> [B*T, dim]
        pred_pos_flat = torch.cat(all_pred_pos, dim=0)
        pred_class_flat = torch.cat(all_pred_class, dim=0)
        pred_state_logits_flat = torch.cat(all_pred_state_logits, dim=0)
        
        # Ground Truths: [B, T, dim] -> [B*T, dim]
        gt_pos_flat = batch['gt_pos'].reshape(B * T, -1).to(DEVICE)
        
        # GT do ESTADO SIMBÓLICO (0, 1, 2, -1) - O QUE É GERADO PELA LÓGICA MEALY
        gt_state_flat = batch.get('gt_state', batch['gt_class']).reshape(B * T).to(DEVICE) 
        
        # GT da CLASSE DO DRONE (0, 1, 2, 3, -1) - O GT RAW DE CLASSIFICAÇÃO
        # Assumimos que o dataloader deve prover uma chave separada
        # Se não houver, voltamos para o gt_state, mas o modelo não aprenderá a classe
        gt_drone_class_flat = batch.get('gt_drone_class', gt_state_flat).reshape(B * T).to(DEVICE) 

        # 1. Loss de Posição (MSE)
        loss_pos = criterion_pos(pred_pos_flat, gt_pos_flat)
        
        # 2. Loss de Estado Simbólico (Mealy State)
        # Usa gt_state_flat (0, 1, 2, -1) para treinar pred_state_logits
        valid_state_mask = gt_state_flat != -1
        if valid_state_mask.any():
            valid_gt_state = gt_state_flat[valid_state_mask]
            valid_pred_state_logits = pred_state_logits_flat[valid_state_mask]
            loss_state = criterion_state(valid_pred_state_logits, valid_gt_state)
        else:
            loss_state = torch.tensor(0.0).to(DEVICE)

        # 3. Loss de Classificação do Drone (Tipo 0, 1, 2, 3)
        # Usa gt_drone_class_flat (0, 1, 2, 3, -1) para treinar pred_class
        valid_cls_mask = gt_drone_class_flat != -1
        if valid_cls_mask.any():
            valid_gt_class = gt_drone_class_flat[valid_cls_mask]
            valid_pred_class = pred_class_flat[valid_cls_mask]
            loss_cls = criterion_cls(valid_pred_class, valid_gt_class)
        else:
            loss_cls = torch.tensor(0.0).to(DEVICE)
        
        # Loss Total: A soma das 3 cabeças.
        # Você pode adicionar pesos de loss aqui, se desejar (e.g., total_loss = loss_pos + 0.5 * loss_cls + 2.0 * loss_state)
        total_loss = loss_pos + loss_cls + loss_state
        
        # Backward pass
        total_loss.backward()
        
        # IMPLEMENTAÇÃO DO GRADIENT CLIPPING
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRADIENT_NORM)
        
        # Otimização
        optimizer.step()
        
        # running_loss deve ser normalizado pela quantidade de amostras no lote, não apenas pelo lote
        running_loss += total_loss.item() * (B * T) 

    # Retorna a média da loss por amostra (total de amostras = len(dataloader.dataset) * sequence_length)
    total_samples = len(dataloader.dataset) * T
    return running_loss / total_samples


def validate_epoch(model, dataloader, criterion_pos, criterion_cls, criterion_state):
    """
    Executa uma época de validação com BPTT.
    """
    model.eval()
    val_loss_pos = 0.0
    val_loss_cls = 0.0
    val_loss_state = 0.0
    correct_cls_predictions = 0
    total_cls_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validação"):
            B, T = batch['image'].shape[:2]
            
            prev_mealy_state = None 
            
            all_pred_pos = []
            all_pred_class = []
            all_pred_state_logits = []

            for t in range(T):
                # Extrai frame t
                image_t = batch['image'][:, t, ...].to(DEVICE)
                lidar_t = batch['lidar'][:, t, ...].to(DEVICE)
                radar_t = batch['radar'][:, t, ...].to(DEVICE)
                audio_t = batch['audio'][:, t, ...].to(DEVICE)
                
                # Forward pass (passando o estado anterior)
                output = model(image_t, lidar_t, radar_t, audio_t, prev_mealy_state)
                
                all_pred_pos.append(output['pred_pos'])
                all_pred_class.append(output['pred_class'])
                all_pred_state_logits.append(output['pred_state_logits'])
                
                # Propaga o estado
                prev_mealy_state = output['next_mealy_state']
            
            # --- Cálculo da Loss (Empilhando as predições de T) ---
            pred_pos_flat = torch.cat(all_pred_pos, dim=0)
            pred_class_flat = torch.cat(all_pred_class, dim=0)
            pred_state_logits_flat = torch.cat(all_pred_state_logits, dim=0)
            
            gt_pos_flat = batch['gt_pos'].reshape(B * T, -1).to(DEVICE)
            gt_state_flat = batch.get('gt_state', batch['gt_class']).reshape(B * T).to(DEVICE)
            gt_drone_class_flat = batch.get('gt_drone_class', gt_state_flat).reshape(B * T).to(DEVICE)

            # 1. Loss de Posição (MSE)
            loss_pos = criterion_pos(pred_pos_flat, gt_pos_flat)
            val_loss_pos += loss_pos.item() * (B * T)

            # 2. Loss de Estado Simbólico (Mealy State)
            valid_state_mask = gt_state_flat != -1
            if valid_state_mask.any():
                valid_gt_state = gt_state_flat[valid_state_mask]
                valid_pred_state_logits = pred_state_logits_flat[valid_state_mask]
                loss_state = criterion_state(valid_pred_state_logits, valid_gt_state)
                val_loss_state += loss_state.item() * valid_gt_state.size(0)

            # 3. Loss de Classificação (Tipo do Drone + Accuracy)
            valid_cls_mask = gt_drone_class_flat != -1
            if valid_cls_mask.any():
                valid_gt_class = gt_drone_class_flat[valid_cls_mask]
                valid_pred_class = pred_class_flat[valid_cls_mask]
                
                loss_cls = criterion_cls(valid_pred_class, valid_gt_class)
                val_loss_cls += loss_cls.item() * valid_gt_class.size(0)

                # Accuracy de Classificação do Drone
                _, predicted_classes = torch.max(valid_pred_class, 1)
                correct_cls_predictions += (predicted_classes == valid_gt_class).sum().item()
                total_cls_samples += valid_gt_class.size(0)
            
    # Métricas de Retorno
    total_samples_time = len(dataloader.dataset) * T
    total_valid_state_samples = total_samples_time - (gt_state_flat == -1).sum().item()
    
    # Perda média por amostra
    avg_loss_pos = val_loss_pos / total_samples_time
    avg_loss_cls = val_loss_cls / (total_cls_samples if total_cls_samples > 0 else 1)
    avg_loss_state = val_loss_state / (total_valid_state_samples if total_valid_state_samples > 0 else 1)
    
    accuracy_cls = (correct_cls_predictions / total_cls_samples) * 100 if total_cls_samples > 0 else 0.0
    
    return avg_loss_pos, avg_loss_cls, avg_loss_state, accuracy_cls


# --- 2. CONFIGURAÇÃO PRINCIPAL E EXECUÇÃO ---

def main():
    
    # 2.1. Carregamento dos Dados
    train_dataset = MMDataset(CONFIG["train_split_path"], CONFIG["data_root"], CONFIG, phase='train')
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG["batch_size"], 
        shuffle=True, 
        num_workers=CONFIG["num_workers"],
        pin_memory=True 
    )
    
    # Para o val, é importante usar o mesmo data_root, mas o split de validação
    val_dataset = MMDataset(CONFIG["val_split_path"], CONFIG["data_root"], CONFIG, phase='val') 
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG["batch_size"], 
        shuffle=False, 
        num_workers=CONFIG["num_workers"],
        pin_memory=True
    )

    # 2.2. Inicialização do Modelo, Otimizador e Losses
    
    # Garante que o dropout está no CONFIG para o modelo, se necessário
    if 'dropout_rate' not in CONFIG:
        CONFIG['dropout_rate'] = 0.1 
    
    model = MultiModalTransformer(CONFIG).to(DEVICE)
    
    # Lembre-se: nn.CrossEntropyLoss lida automaticamente com logits
    criterion_pos = nn.MSELoss() 
    # Usado para TIPO DE DRONE (4 classes: 0, 1, 2, 3)
    criterion_cls = nn.CrossEntropyLoss(ignore_index=-1) 
    # Usado para ESTADO SIMBÓLICO (3 classes: 0, 1, 2)
    criterion_state = nn.CrossEntropyLoss(ignore_index=-1) 
    
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    
    # 2.3. Loop Principal de Treinamento
    
    best_val_loss = float('inf')
    
    for epoch in range(CONFIG["epochs"]):
        print(f"\n--- Época {epoch+1}/{CONFIG['epochs']} ---")
        
        # Treinamento
        train_loss = train_epoch(model, train_loader, criterion_pos, criterion_cls, criterion_state, optimizer)
        print(f"Média da Loss de Treinamento (Total): {train_loss:.4f}")
        
        # Validação
        val_loss_pos, val_loss_cls, val_loss_state, val_accuracy_cls = validate_epoch(model, val_loader, criterion_pos, criterion_cls, criterion_state)
        
        total_val_loss = val_loss_pos + val_loss_cls + val_loss_state
        
        print(f"\n[Validação] Loss Total: {total_val_loss:.4f}")
        print(f"  > Loss Posição (MSE): {val_loss_pos:.4f}")
        print(f"  > Loss Classificação Drone: {val_loss_cls:.4f}")
        print(f"  > Loss Estado Simbólico (Mealy): {val_loss_state:.4f}")
        print(f"  > Accuracy Classificação Drone: {val_accuracy_cls:.2f}%")
        
        # Salva o melhor modelo (Critério: Menor Total Loss de Validação)
        if total_val_loss < best_val_loss:
            best_val_loss = total_val_loss
            torch.save(model.state_dict(), 'best_mm_transformer_mealey_model.pth')
            print(">>> Modelo salvo: Melhor resultado de Validação <<<")

    print("\nTreinamento concluído.")

if __name__ == "__main__":
    # Certifique-se de que o DATA_ROOT está configurado corretamente em mm_config.py
    main()


# Teste

In [5]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from mm_config_mealey import CONFIG
from mm_transformer_mealey import MultiModalTransformer
from PIL import Image

# =========================================================
#  CONFIGURAÇÃO DE AVALIAÇÃO (usa CSV de referência)
# =========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VAL_DIR = os.path.join(CONFIG["data_root"], "val")
VAL_GT_CSV = os.path.join(VAL_DIR, "validation_ref.csv")

# =========================================================
#  CARREGAR GT DO CSV
# =========================================================
df_gt = pd.read_csv(VAL_GT_CSV)
df_gt["Position"] = df_gt["Position"].apply(lambda x: np.array(eval(x), dtype=np.float32))

# Agrupa por sequência
grouped_gt = {seq: seq_df for seq, seq_df in df_gt.groupby("Sequence")}
available_sequences = sorted(grouped_gt.keys())

print(f"Encontradas {len(available_sequences)} sequências no CSV: {available_sequences}")

# =========================================================
#  INICIALIZAR MODELO
# =========================================================
model = MultiModalTransformer(CONFIG).to(DEVICE)
model.eval()

# Se houver checkpoint salvo, descomente:
# model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))

# =========================================================
#  FUNÇÃO AUXILIAR PARA CARREGAR FEATURES
# =========================================================
def load_feature(base_path, timestamp_id, feature_dim=CONFIG["modal_feature_dim"]):
    npy_path = os.path.join(base_path, f"{timestamp_id}.npy")
    if not os.path.exists(npy_path):
        return torch.zeros(feature_dim, dtype=torch.float32)
    arr = np.load(npy_path, allow_pickle=True)
    arr = arr.flatten()
    if arr.shape[0] < feature_dim:
        arr = np.pad(arr, (0, feature_dim - arr.shape[0]))
    else:
        arr = arr[:feature_dim]
    return torch.tensor(arr, dtype=torch.float32)

# =========================================================
#  LOOP DE AVALIAÇÃO
# =========================================================
all_gt_pos, all_pred_pos = [], []
all_gt_class, all_pred_class = [], []
all_pred_symbolic, all_gt_symbolic = [], []

with torch.no_grad():
    for seq in tqdm(available_sequences, desc="Avaliando VAL (usando CSV)"):
        seq_path = os.path.join(VAL_DIR, seq)
        seq_df = grouped_gt[seq]

        prev_state = None
        for _, row in seq_df.iterrows():
            ts = str(row["Timestamp"])
            gt_pos = row["Position"]
            gt_class = int(row["Classification"])

            # Caminhos base
            lidar_path = os.path.join(seq_path, "lidar_360")
            radar_path = os.path.join(seq_path, "radar_enhance_pcl")
            audio_path = os.path.join(CONFIG["data_root"], "audio_features", seq)

            # =========================
            # Carrega imagem usando prefixo do timestamp
            # =========================
            try:
                ts_prefix = ts.split(".")[0]
                img_dir = os.path.join(seq_path, "Image")
                img_files = [f for f in os.listdir(img_dir) if f.startswith(ts_prefix)]
                if len(img_files) > 0:
                    img_file = img_files[0]
                    img_path = os.path.join(img_dir, img_file)
                    img = Image.open(img_path).convert("RGB").resize((CONFIG["image_size"], CONFIG["image_size"]))
                    img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
                else:
                    img_tensor = torch.zeros((3, CONFIG["image_size"], CONFIG["image_size"]))
            except Exception:
                img_tensor = torch.zeros((3, CONFIG["image_size"], CONFIG["image_size"]))

            # Carrega demais features
            lidar = load_feature(lidar_path, ts)
            radar = load_feature(radar_path, ts)
            audio = load_feature(audio_path, ts)

            # Envia para GPU
            img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
            lidar = lidar.unsqueeze(0).to(DEVICE)
            radar = radar.unsqueeze(0).to(DEVICE)
            audio = audio.unsqueeze(0).to(DEVICE)

            # Inferência
            outputs = model(
                img_tensor,
                lidar,
                radar,
                audio,
                prev_mealy_state=prev_state
            )
            prev_state = outputs["next_mealy_state"]

            pred_pos = outputs["pred_pos"].cpu().numpy().flatten()
            pred_class = torch.argmax(outputs["pred_class"], dim=1).cpu().item()
            pred_state = torch.argmax(outputs["pred_state_logits"], dim=1).cpu().item()

            # Armazena resultados
            all_gt_pos.append(gt_pos)
            all_pred_pos.append(pred_pos)
            all_gt_class.append(gt_class)
            all_pred_class.append(pred_class)
            all_pred_symbolic.append(pred_state)
            all_gt_symbolic.append(gt_class)

# =========================================================
#  CÁLCULO DAS MÉTRICAS
# =========================================================
all_gt_pos = np.array(all_gt_pos)
all_pred_pos = np.array(all_pred_pos)

rmse_xyz = np.sqrt(mean_squared_error(all_gt_pos, all_pred_pos, multioutput="raw_values"))
rmse_total = np.sqrt(mean_squared_error(all_gt_pos.flatten(), all_pred_pos.flatten()))
acc_class = accuracy_score(all_gt_class, all_pred_class)
conf_mat = confusion_matrix(all_gt_symbolic, all_pred_symbolic)

print("\n=== RESULTADOS DE DESEMPENHO (VAL - CSV) ===")
print(f"RMSE X: {rmse_xyz[0]:.4f}, Y: {rmse_xyz[1]:.4f}, Z: {rmse_xyz[2]:.4f}")
print(f"RMSE TOTAL: {rmse_total:.4f}")
print(f"Acurácia de Classificação: {acc_class*100:.2f}%")
print("\nMatriz de Confusão dos Estados Simbólicos:")
print(conf_mat)

# =========================================================
#  SALVAR RESULTADOS DETALHADOS
# =========================================================
out_df = pd.DataFrame({
    "gt_x": all_gt_pos[:, 0],
    "gt_y": all_gt_pos[:, 1],
    "gt_z": all_gt_pos[:, 2],
    "pred_x": all_pred_pos[:, 0],
    "pred_y": all_pred_pos[:, 1],
    "pred_z": all_pred_pos[:, 2],
    "gt_class": all_gt_class,
    "pred_class": all_pred_class,
    "gt_symbolic": all_gt_symbolic,
    "pred_symbolic": all_pred_symbolic
})
out_df.to_csv("val_predictions_csv_results.csv", index=False)
print("\nResultados detalhados salvos em 'val_predictions_csv_results.csv'.")


Encontradas 16 sequências no CSV: ['seq0001', 'seq0002', 'seq0003', 'seq0004', 'seq0005', 'seq0006', 'seq0007', 'seq0008', 'seq0009', 'seq0010', 'seq0011', 'seq0012', 'seq0013', 'seq0014', 'seq0015', 'seq0016']


Avaliando VAL (usando CSV): 100%|██████████| 16/16 [01:29<00:00,  5.59s/it]


=== RESULTADOS DE DESEMPENHO (VAL - CSV) ===
RMSE X: 5.4672, Y: 7.4411, Z: 28.4489
RMSE TOTAL: 17.2684
Acurácia de Classificação: 68.19%

Matriz de Confusão dos Estados Simbólicos:
[[200   0   0   0]
 [600   0   0   0]
 [700   0   0   0]
 [100   0   0   0]]

Resultados detalhados salvos em 'val_predictions_csv_results.csv'.


# integridade do teste

In [6]:
import os
import pandas as pd
from tqdm import tqdm
from mm_config_mealey import CONFIG

VAL_DIR = os.path.join(CONFIG["data_root"], "val")
CSV_PATH = os.path.join(VAL_DIR, "validation_ref.csv")

print(f"\nVerificando integridade dos dados em:\n{VAL_DIR}")
df = pd.read_csv(CSV_PATH)
df["Timestamp"] = df["Timestamp"].astype(str)

records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Verificando arquivos"):
    seq = row["Sequence"]
    ts = row["Timestamp"]

    seq_path = os.path.join(VAL_DIR, seq)
    checks = {
        "image": os.path.exists(os.path.join(seq_path, "Image", f"{ts}.png")),
        "lidar": os.path.exists(os.path.join(seq_path, "lidar_360", f"{ts}.npy")),
        "livox": os.path.exists(os.path.join(seq_path, "livox_avia", f"{ts}.npy")),
        "radar": os.path.exists(os.path.join(seq_path, "radar_enhance_pcl", f"{ts}.npy")),
    }

    # Modalidades disponíveis (True)
    available = [k for k, v in checks.items() if v]
    missing = [k for k, v in checks.items() if not v]

    records.append({
        "Sequence": seq,
        "Timestamp": ts,
        **checks,
        "available_modalities": ",".join(available),
        "missing_modalities": ",".join(missing)
    })

report_df = pd.DataFrame(records)

total = len(report_df)
missing_any = report_df[report_df["missing_modalities"].apply(lambda x: len(x) > 0)]
pct_missing = (len(missing_any) / total) * 100

print(f"\nTotal de amostras: {total}")
print(f"Amostras com algum arquivo faltando: {len(missing_any)} ({pct_missing:.2f}%)")

report_df.to_csv("val_integrity_report_fixed.csv", index=False)
print("\nRelatório salvo em 'val_integrity_report_fixed.csv'.")



Verificando integridade dos dados em:
C:\Users\Micro\Documents\sourcecode\MMNTT\val


Verificando arquivos: 100%|██████████| 1600/1600 [00:00<00:00, 6672.67it/s]


Total de amostras: 1600
Amostras com algum arquivo faltando: 1600 (100.00%)

Relatório salvo em 'val_integrity_report_fixed.csv'.


In [7]:
# =========================================================
#  VISUALIZAÇÕES AUTOMÁTICAS
# =========================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import numpy.linalg as LA

# --- 3D Scatter: Posições ---
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(all_gt_pos[:,0], all_gt_pos[:,1], all_gt_pos[:,2], c='blue', s=5, label='GT')
ax.scatter(all_pred_pos[:,0], all_pred_pos[:,1], all_pred_pos[:,2], c='red', s=5, label='Pred')
ax.set_title("Posições 3D: Ground Truth vs Predição")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend()
plt.tight_layout()
plt.savefig("plot_3d_positions.png", dpi=200)
plt.close()

# --- Histograma de erro por eixo ---
err_xyz = np.abs(all_gt_pos - all_pred_pos)
plt.figure(figsize=(8,5))
plt.hist(err_xyz[:,0], bins=30, alpha=0.5, label='Erro X')
plt.hist(err_xyz[:,1], bins=30, alpha=0.5, label='Erro Y')
plt.hist(err_xyz[:,2], bins=30, alpha=0.5, label='Erro Z')
plt.xlabel("Erro absoluto (metros)")
plt.ylabel("Frequência")
plt.title("Distribuição do erro por eixo")
plt.legend()
plt.tight_layout()
plt.savefig("plot_error_histogram.png", dpi=200)
plt.close()

# --- Curva temporal do erro total ---
err_total = LA.norm(all_gt_pos - all_pred_pos, axis=1)
plt.figure(figsize=(10,4))
plt.plot(err_total)
plt.xlabel("Índice temporal (frames)")
plt.ylabel("Erro Euclidiano (m)")
plt.title("Erro de posição ao longo do tempo")
plt.tight_layout()
plt.savefig("plot_error_over_time.png", dpi=200)
plt.close()

# --- Matriz de confusão normalizada ---
plt.figure(figsize=(5,4))
cm = confusion_matrix(all_gt_symbolic, all_pred_symbolic, normalize='true')
sns.heatmap(cm, annot=True, fmt=".2f", cmap='Blues')
plt.title("Matriz de confusão normalizada (estados simbólicos)")
plt.xlabel("Predição")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.savefig("plot_confusion_matrix.png", dpi=200)
plt.close()

print("\nGráficos salvos:")
print(" - plot_3d_positions.png")
print(" - plot_error_histogram.png")
print(" - plot_error_over_time.png")
print(" - plot_confusion_matrix.png")



Gráficos salvos:
 - plot_3d_positions.png
 - plot_error_histogram.png
 - plot_error_over_time.png
 - plot_confusion_matrix.png
